In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, StratifiedKFold

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score, precision_score, recall_score, f1_score

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectFromModel

from sklearn.pipeline import Pipeline

from collections import Counter
import matplotlib.pyplot as plt

import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from preprocessing_utils import *
import config

In [2]:
X_train = joblib.load("../data/no_boot/X_train.pkl")
X_test = joblib.load("../data/no_boot/X_test.pkl")
y_train = joblib.load("../data/no_boot/y_train.pkl")
y_test = joblib.load("../data/no_boot/y_test.pkl")

In [3]:
def run_LASSO():
    print("Running Logistic Regression with LASSO")

    pipeline = Pipeline([
        ('var_thresh', VarianceThreshold(threshold=0.001).set_output(transform="pandas")),
        # ("selector", StabilitySelection()),
        # ("selector", BootstrappedSelectKBest()),
        ('select', SelectKBest(score_func=f_classif, k=50)),
        ('clf', LogisticRegression(
            penalty='l1',
            solver='saga',
            class_weight='balanced',      # helps with imbalance
            random_state=config.SEED,
            max_iter=20000,               # more iterations for convergence
            n_jobs=-1,                    # parallelize
            tol=1e-3,
            verbose=0
        ))
    ])

    param_grid = {
        # Regularization strength (C smaller = stronger shrinkage)
        'clf__C': [0.1, 1, 10]
    }

    return pipeline, param_grid


In [4]:
pipeline, param_grid = run_LASSO()

# Set up cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=config.SEED)

# Grid search over pipeline
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',  # could experiment with 'f1' if recurrence class is more important
    n_jobs=1,
    verbose=3
)

# # Fit pipeline on training data
grid_search = grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_


Running Logistic Regression with LASSO
Fitting 3 folds for each of 3 candidates, totalling 9 fits
[CV 1/3] END ........................clf__C=0.1;, score=0.663 total time=   0.5s
[CV 2/3] END ........................clf__C=0.1;, score=0.645 total time=   0.4s
[CV 3/3] END ........................clf__C=0.1;, score=0.527 total time=   0.3s
[CV 1/3] END ..........................clf__C=1;, score=0.654 total time=   0.4s
[CV 2/3] END ..........................clf__C=1;, score=0.643 total time=   0.4s
[CV 3/3] END ..........................clf__C=1;, score=0.503 total time=   1.8s
[CV 1/3] END .........................clf__C=10;, score=0.640 total time=   0.5s
[CV 2/3] END .........................clf__C=10;, score=0.603 total time=   0.4s
[CV 3/3] END .........................clf__C=10;, score=0.470 total time=   0.4s


In [5]:
joblib.dump(best_model, "../models/testing_LASSO.pkl")

['../models/testing_LASSO.pkl']